In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!unrar x "/content/drive/MyDrive/processed.rar" "/content/"

Streaming output truncated to the last 5000 lines.
Extracting  /content/processed/svhn_number_yolo/labels/train/8351.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8352.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8354.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8355.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8357.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8358.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8359.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/836.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8360.txt      99%  OK 
Extracting  /content/processed/svhn_number_yolo/labels/train/8361.txt      99%  OK 
Extracting  /content

In [14]:
from pathlib import Path

YOLO_DIR = Path("/content/processed/svhn_number_yolo")

print("Exists:", YOLO_DIR.exists())

print("Train images:", len(list((YOLO_DIR / "images/train").glob("*.png"))))
print("Train labels:", len(list((YOLO_DIR / "labels/train").glob("*.txt"))))

print("Val images:", len(list((YOLO_DIR / "images/val").glob("*.png"))))
print("Val labels:", len(list((YOLO_DIR / "labels/val").glob("*.txt"))))

print("Test images:", len(list((YOLO_DIR / "images/test").glob("*.png"))))
print("Test labels:", len(list((YOLO_DIR / "labels/test").glob("*.txt"))))

Exists: True
Train images: 30061
Train labels: 30061
Val images: 3341
Val labels: 3341
Test images: 13068
Test labels: 13068


In [15]:
yaml_text = f"""
path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: number
""".strip()

yaml_path = YOLO_DIR / "svhn_number_colab.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print(yaml_path.read_text())

path: /content/processed/svhn_number_yolo
train: images/train
val: images/val
test: images/test

names:
  0: number


In [22]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=416,
    batch=16,
    project="/content/drive/MyDrive/lab5_runs",
    name="svhn_number_detector_test_fast",
    device=0
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/processed/svhn_number_yolo/svhn_number_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=svhn_number_detector_test_fast, nbs=64, nms=False, opset=None, optimize=False

In [24]:
from ultralytics import YOLO

best_model = YOLO("/content/drive/MyDrive/lab5_runs/svhn_number_detector_test_fast/weights/best.pt")

metrics = best_model.val(
    data=str(yaml_path),
    split="test"
)

print(metrics)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 176.9±282.5 MB/s, size: 41.8 KB)
val: Scanning /content/processed/svhn_number_yolo/labels/test... 13068 images, 0 backgrounds, 4 corrupt: 100% ━━━━━━━━━━━━ 13068/13068 1.6Kit/s 8.4s
val: /content/processed/svhn_number_yolo/images/test/3192.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1818]
val: /content/processed/svhn_number_yolo/images/test/3940.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0811]
val: /content/processed/svhn_number_yolo/images/test/7267.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0606]
val: /content/processed/svhn_number_yolo/images/test/8690.png: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0789]
v

In [25]:
sample_test_images = list((YOLO_DIR / "images/test").glob("*.png"))[:8]

best_model.predict(
    source=[str(p) for p in sample_test_images],
    save=True,
    conf=0.25
)


0: 416x416 1 number, 9.8ms
1: 416x416 1 number, 9.8ms
2: 416x416 1 number, 9.8ms
3: 416x416 1 number, 9.8ms
4: 416x416 1 number, 9.8ms
5: 416x416 1 number, 9.8ms
6: 416x416 1 number, 9.8ms
7: 416x416 1 number, 9.8ms
Speed: 2.3ms preprocess, 9.8ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 416)
Results saved to /content/runs/detect/predict


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'number'}
 obb: None
 orig_img: array([[[82, 85, 83],
         [82, 85, 83],
         [82, 83, 81],
         ...,
         [83, 83, 83],
         [83, 83, 83],
         [83, 83, 83]],
 
        [[83, 86, 84],
         [83, 86, 84],
         [84, 85, 83],
         ...,
         [84, 84, 84],
         [83, 83, 83],
         [83, 83, 83]],
 
        [[83, 86, 84],
         [83, 86, 84],
         [84, 85, 83],
         ...,
         [84, 84, 84],
         [83, 83, 83],
         [83, 83, 83]],
 
        ...,
 
        [[92, 92, 92],
         [92, 93, 91],
         [92, 93, 91],
         ...,
         [94, 93, 89],
         [92, 93, 89],
         [91, 92, 88]],
 
        [[95, 93, 92],
         [95, 94, 90],
         [95, 94, 90],
         ...,
         [96, 96, 90],
         [93, 95, 89],
         [93, 95, 89]],
 
        [[94, 94, 94],
    